# P1 · ¿Qué tiempo hizo?

Datos diarios de la estación **A Coruña / Alvedro (aeropuerto)**, indicativo **08002**, del 1 de enero de 2021 al 31 de diciembre de 2025.

Pasos:

1. Descargar y guardar el fichero
2. Mirar el fichero antes de leerlo
3. Leer con pandas
4. Gráfica antes de limpiar
5. Limpiar
6. Guardar los datos limpios
7. Analizar
8. Gráficas
9. Eficiencia
10. PostgreSQL

> Ejecutar desde la carpeta `notebooks/` con *Kernel → Restart & Run All*.

## Paso 1. Descargar y guardar el fichero

Se descarga desde código y se guarda **sin modificar** en `datos/raw/` (se escriben los bytes tal cual llegan). Si las carpetas no existen, se crean.

In [ ]:
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

URL = 'https://raw.githubusercontent.com/bigdatawirtz/datasets/main/datasets/meteo_alvedro_2021_2025.csv'
RAIZ = Path('..')
RAW = RAIZ / 'datos' / 'raw' / 'meteo_alvedro_2021_2025.csv'
CLEAN = RAIZ / 'datos' / 'clean' / 'meteo_alvedro_limpio.csv'
OUTPUT = RAIZ / 'output'

for carpeta in [RAW.parent, CLEAN.parent, OUTPUT]:
    carpeta.mkdir(parents=True, exist_ok=True)

respuesta = requests.get(URL, timeout=60)
respuesta.raise_for_status()
RAW.write_bytes(respuesta.content)
print('Guardado en', RAW, '-', RAW.stat().st_size, 'bytes')
print('Fecha de descarga:', date.today())

## Paso 2. Mirar el fichero antes de leerlo

Primero los bytes, para saber la codificación.

In [ ]:
contenido = RAW.read_bytes()
print(contenido[:200])

try:
    contenido.decode('utf-8')
    print('El fichero es UTF-8')
except UnicodeDecodeError as error:
    print('El fichero NO es UTF-8:', error)

No es UTF-8: letras como `ó` aparecen como un solo byte (`\xf3`), así que la codificación es **latin-1**.

Ahora las primeras y últimas líneas.

In [ ]:
lineas = contenido.decode('latin-1').splitlines()

print('--- Primeras líneas ---')
for linea in lineas[:10]:
    print(linea)

print('\n--- Últimas líneas ---')
for linea in lineas[-8:]:
    print(linea)

**Lo que se ve:**

- Codificación **latin-1**.
- Las **6 primeras líneas no son datos** (5 de metadatos y 1 en blanco). La cabecera está en la línea 7.
- Separador de campos **`;`** y separador decimal **`,`**.
- La presión usa **punto de miles** (`1.015,7`).
- Al final aparecen filas de fechas antiguas: posibles **duplicados** y filas desordenadas.

Los datos de la estación están en los metadatos:

In [ ]:
for linea in lineas[1:4]:
    print(linea)

## Paso 3. Leer con pandas

Primera lectura con lo que ya sabemos, para ver los tipos de las columnas.

In [ ]:
COLUMNAS = ['fecha', 'tmed', 'tmin', 'tmax', 'hum', 'prec', 'viento', 'pres']

df = pd.read_csv(RAW, sep=';', encoding='latin-1', skiprows=6, decimal=',')
df.columns = COLUMNAS
df.dtypes

`prec` y `pres` deberían ser números, pero se leen como texto (`object`). Veamos por qué:

In [ ]:
print('Valores de prec:', df['prec'].unique()[:20])
print('Valores de pres:', df['pres'].head().tolist())
print()
df.describe()

- **`prec`** tiene el valor `Ip` (*inapreciable*: llovió menos de 0,1 mm). Un solo texto convierte toda la columna en texto.
- **`pres`** tiene el punto de miles (`1.015,7`), que pandas no entiende si no se le indica.
- En `describe()` el mínimo de las temperaturas es **-9999**: es un código de valor ausente, no una temperatura.
- También hay ausentes como `NA` y campos vacíos (pandas ya los lee como `NaN`).

Lectura definitiva:

- `thousands='.'` para la presión.
- `na_values=['NA', '-9999']` para los ausentes.
- `Ip` se cambia por `0`: no es un dato que falte, es lluvia tan pequeña que no se pudo medir.

In [ ]:
df = pd.read_csv(RAW, sep=';', encoding='latin-1', skiprows=6, decimal=',', thousands='.',
                 na_values=['NA', '-9999'])
df.columns = COLUMNAS

print('Valores Ip:', (df['prec'] == 'Ip').sum())
df['prec'] = pd.to_numeric(df['prec'].replace('Ip', '0').str.replace(',', '.'))

La fecha viene en **dos formatos**: `AAAA-MM-DD` y, en parte de 2023, `DD/MM/AAAA`. Se convierte cada formato por separado (si dejamos que pandas adivine, puede confundir el día con el mes).

In [ ]:
fecha_iso = pd.to_datetime(df['fecha'], format='%Y-%m-%d', errors='coerce')
fecha_europea = pd.to_datetime(df['fecha'], format='%d/%m/%Y', errors='coerce')
print('Filas con fecha DD/MM/AAAA:', fecha_europea.notna().sum())

df['fecha'] = fecha_iso.fillna(fecha_europea)
print('Fechas sin convertir:', df['fecha'].isna().sum())
df.dtypes

Ahora todas las columnas tienen el tipo correcto.

## Paso 4. Gráfica antes de limpiar

In [ ]:
orden = df.sort_values('fecha')

plt.figure(figsize=(14, 5))
plt.plot(orden['fecha'], orden['tmax'], '.', ms=3, color='tab:red', label='tmax')
plt.plot(orden['fecha'], orden['tmed'], '.', ms=3, color='tab:green', label='tmed')
plt.plot(orden['fecha'], orden['tmin'], '.', ms=3, color='tab:blue', label='tmin')
plt.title('Temperaturas antes de limpiar')
plt.xlabel('Fecha')
plt.ylabel('Temperatura (°C)')
plt.legend()
plt.show()

En la gráfica se ven **puntos sueltos que no encajan** (más de 30 °C en invierno, una mínima bajo cero en septiembre) y un **hueco en julio de 2023** (faltan días).

## Paso 5. Limpiar

### 5.1 Valores ausentes

In [ ]:
df.isna().sum()

**Decisión:** son muy pocos (alrededor del 1 % por variable), así que **se dejan como `NaN`**. No se rellenan porque sería inventar datos, y no se borra la fila entera para no perder las demás variables de ese día. Las medias y sumas de pandas y de SQL ignoran los valores vacíos.

### 5.2 Duplicados y orden

In [ ]:
print('Filas duplicadas:', df.duplicated().sum())
print('¿Ordenado por fecha?', df['fecha'].is_monotonic_increasing)

df = df.drop_duplicates().sort_values('fecha').reset_index(drop=True)
print('Filas tras eliminar duplicados:', len(df))

### 5.3 Días que faltan

Se compara con un calendario completo de 2021 a 2025.

In [ ]:
calendario = pd.date_range('2021-01-01', '2025-12-31')
faltan = calendario.difference(df['fecha'])
print('Días que faltan:', len(faltan))
print(faltan.date)

# Se añaden como filas vacías para que los huecos se vean en las gráficas
df = df.set_index('fecha').reindex(calendario).rename_axis('fecha').reset_index()
print('Filas con el calendario completo:', len(df))

Faltan 27 días, la mayoría seguidos en **julio de 2023**. No se inventan: quedan como filas vacías.

### 5.4 Valores fuera de rango e inconsistentes

Se comprueba que los valores sean posibles (humedad entre 0 y 100, lluvia no negativa, mínima no mayor que la máxima) y que la temperatura media encaje con la mínima y la máxima. La media de un día debe estar cerca de `(tmin + tmax) / 2`; si se aleja más de 6 °C, alguna de las tres temperaturas está mal.

In [ ]:
print('Humedad fuera de 0-100:', ((df['hum'] < 0) | (df['hum'] > 100)).sum())
print('Lluvia negativa:', (df['prec'] < 0).sum())
print('tmin mayor que tmax:', (df['tmin'] > df['tmax']).sum())

no_encaja = (df['tmed'] - (df['tmin'] + df['tmax']) / 2).abs() > 6
print('Días con temperaturas que no encajan:', no_encaja.sum())
df[no_encaja]

Son errores claros (por ejemplo, 39,2 °C de máxima en marzo con 10,6 °C de media). Como no se sabe cuál de las tres temperaturas es la buena, **se anulan las temperaturas de esos días**.

In [ ]:
df.loc[no_encaja, ['tmed', 'tmin', 'tmax']] = np.nan
df.describe()

## Paso 6. Guardar los datos limpios

En `datos/clean/` y en UTF-8.

In [ ]:
df.to_csv(CLEAN, index=False, encoding='utf-8')
print('Guardado en', CLEAN)

## Paso 7. Analizar

In [ ]:
df['anio'] = df['fecha'].dt.year
df['mes'] = df['fecha'].dt.month
MESES = ['enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio', 'julio', 'agosto',
         'septiembre', 'octubre', 'noviembre', 'diciembre']

### Pregunta 1. ¿Cuál fue la temperatura media de cada año? ¿Hay tendencia?

In [ ]:
media_anual = df.groupby('anio')['tmed'].mean()
print(media_anual.round(2))
print()
print(f'Interpretación: la media anual va de {media_anual.min():.2f} °C ({media_anual.idxmin()}) '
      f'a {media_anual.max():.2f} °C ({media_anual.idxmax()}). La diferencia es pequeña y con solo 5 años '
      f'no se puede hablar de una tendencia.')

### Pregunta 2. ¿Cuál es, de media, el mes más lluvioso y el más seco?

Se suma la lluvia de cada mes de cada año y después se hace la media de cada mes. Nota: julio de 2023 está incompleto, así que su total sale más bajo de lo real.

In [ ]:
lluvia_mensual = df.groupby(['anio', 'mes'])['prec'].sum(min_count=1)
lluvia_media_mes = lluvia_mensual.groupby('mes').mean()
print(lluvia_media_mes.round(1))
print()
print(f'Interpretación: el mes más lluvioso es {MESES[lluvia_media_mes.idxmax() - 1]} '
      f'({lluvia_media_mes.max():.1f} mm) y el más seco {MESES[lluvia_media_mes.idxmin() - 1]} '
      f'({lluvia_media_mes.min():.1f} mm).')

### Pregunta 3. ¿Cuántos días de lluvia hay cada mes?

**Criterio:** día de lluvia = día con **1 mm o más**.

In [ ]:
df['dia_lluvia'] = df['prec'] >= 1
dias_lluvia = df.groupby(['anio', 'mes'])['dia_lluvia'].sum().unstack()
display(dias_lluvia)

media_dias = dias_lluvia.mean()
print(f'Interpretación: de media, {MESES[media_dias.idxmax() - 1]} es el mes con más días de lluvia '
      f'({media_dias.max():.1f}) y {MESES[media_dias.idxmin() - 1]} el que menos ({media_dias.min():.1f}).')

### Pregunta 4. ¿Cuál fue el día más cálido y el más frío de cada año?

Más cálido según la temperatura máxima y más frío según la mínima.

In [ ]:
mas_calido = df.loc[df.groupby('anio')['tmax'].idxmax(), ['anio', 'fecha', 'tmax']]
mas_frio = df.loc[df.groupby('anio')['tmin'].idxmin(), ['anio', 'fecha', 'tmin']]

extremos = mas_calido.merge(mas_frio, on='anio', suffixes=('_calido', '_frio'))
display(extremos)
print('Interpretación: los días más cálidos caen en verano y los más fríos en invierno, '
      'con una diferencia moderada propia del clima de costa.')

## Paso 8. Gráficas

### Temperatura diaria con media móvil de 30 días

In [ ]:
media_movil = df['tmed'].rolling(30, center=True, min_periods=15).mean()

plt.figure(figsize=(14, 5))
plt.plot(df['fecha'], df['tmed'], color='lightgray', label='Temperatura media diaria')
plt.plot(df['fecha'], media_movil, color='tab:red', label='Media móvil 30 días')
plt.title('Temperatura media diaria en A Coruña / Alvedro')
plt.xlabel('Fecha')
plt.ylabel('Temperatura (°C)')
plt.legend()
plt.savefig(OUTPUT / '01_temperatura_diaria.png')
plt.show()

### Precipitación mensual

In [ ]:
lluvia_por_mes = df.set_index('fecha')['prec'].resample('MS').sum()

plt.figure(figsize=(14, 5))
plt.bar(lluvia_por_mes.index, lluvia_por_mes.values, width=25, color='tab:blue')
plt.title('Precipitación mensual en A Coruña / Alvedro')
plt.xlabel('Mes')
plt.ylabel('Precipitación (mm)')
plt.savefig(OUTPUT / '02_precipitacion_mensual.png')
plt.show()

### Último año comparado con la media de los anteriores

In [ ]:
temp_mensual = df.groupby(['anio', 'mes'])['tmed'].mean().unstack('anio')
ultimo = temp_mensual.columns.max()

plt.figure(figsize=(12, 5))
plt.plot(temp_mensual.index, temp_mensual.drop(columns=ultimo).mean(axis=1), 'o--', color='gray', label='Media 2021-2024')
plt.plot(temp_mensual.index, temp_mensual[ultimo], 'o-', color='tab:red', label=str(ultimo))
plt.xticks(range(1, 13), [m[:3] for m in MESES])
plt.title(f'Temperatura media mensual: {ultimo} frente a la media de los años anteriores')
plt.xlabel('Mes')
plt.ylabel('Temperatura media (°C)')
plt.legend()
plt.savefig(OUTPUT / '03_ultimo_anio.png')
plt.show()

## Paso 9. Eficiencia

Amplitud térmica diaria (`tmax - tmin`) con un bucle `for` y con una operación vectorizada.

In [ ]:
def amplitud_bucle(datos):
    resultado = []
    for _, fila in datos.iterrows():
        resultado.append(fila['tmax'] - fila['tmin'])
    return resultado

def amplitud_vectorizada(datos):
    return datos['tmax'] - datos['tmin']

print(f'DataFrame original: {len(df)} filas')
%timeit amplitud_bucle(df)
%timeit amplitud_vectorizada(df)

In [ ]:
grande = pd.concat([df] * 5, ignore_index=True)

print(f'DataFrame 5 veces más grande: {len(grande)} filas')
%timeit amplitud_bucle(grande)
%timeit amplitud_vectorizada(grande)

**Qué se observa:** la versión vectorizada es cientos de veces más rápida. Con 5 veces más filas, el bucle tarda unas 5 veces más, mientras que la versión vectorizada casi no cambia. El bucle procesa las filas una a una en Python, y la operación vectorizada hace la resta de toda la columna de una vez con código compilado de NumPy.

## Paso 10. PostgreSQL

**Antes:** tener Docker Desktop en marcha y el fichero `.env` en la raíz del proyecto (si no existe, copiar `.env.example` como `.env`).

Levantar PostgreSQL con Docker:

In [ ]:
!docker compose -f ../docker-compose.yml up -d

Conectar con las credenciales del `.env` y cargar la tabla limpia con `to_sql`:

In [ ]:
import os
import time
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv(RAIZ / '.env')
motor = create_engine(
    f"postgresql+psycopg2://{os.environ['POSTGRES_USER']}:{os.environ['POSTGRES_PASSWORD']}"
    f"@{os.getenv('POSTGRES_HOST', 'localhost')}:{os.getenv('POSTGRES_PORT', '5432')}/{os.environ['POSTGRES_DB']}"
)

# PostgreSQL tarda unos segundos en arrancar: se reintenta la conexión
for intento in range(30):
    try:
        with motor.connect():
            print('Conectado a PostgreSQL')
        break
    except Exception:
        time.sleep(2)

limpio = pd.read_csv(CLEAN, parse_dates=['fecha'])
limpio.to_sql('meteo', motor, if_exists='replace', index=False)
print('Filas cargadas:', pd.read_sql('SELECT COUNT(*) FROM meteo', motor).iloc[0, 0])

### SQL · Pregunta 1: temperatura media de cada año

In [ ]:
q1 = pd.read_sql('''
    SELECT EXTRACT(YEAR FROM fecha) AS anio, AVG(tmed) AS tmed_media
    FROM meteo
    GROUP BY anio
    ORDER BY anio
''', motor)
display(q1)
print('¿Coincide con pandas?', np.allclose(q1['tmed_media'], media_anual.values))

### SQL · Pregunta 2: lluvia media de cada mes

In [ ]:
q2 = pd.read_sql('''
    WITH mensual AS (
        SELECT EXTRACT(YEAR FROM fecha) AS anio,
               EXTRACT(MONTH FROM fecha) AS mes,
               SUM(prec) AS total
        FROM meteo
        GROUP BY anio, mes
    )
    SELECT mes, AVG(total) AS lluvia_media
    FROM mensual
    GROUP BY mes
    ORDER BY mes
''', motor)
display(q2)
print('¿Coincide con pandas?', np.allclose(q2['lluvia_media'], lluvia_media_mes.values))

### SQL · Pregunta nueva: ¿cuántos días de calor (máxima de 25 °C o más) hubo cada año?

In [ ]:
q3 = pd.read_sql('''
    SELECT EXTRACT(YEAR FROM fecha) AS anio,
           COUNT(*) FILTER (WHERE tmax >= 25) AS dias_calor
    FROM meteo
    GROUP BY anio
    ORDER BY anio
''', motor)
display(q3)

dias_calor_pandas = (df['tmax'] >= 25).groupby(df['anio']).sum()
print('¿Coincide con pandas?', (q3['dias_calor'].values == dias_calor_pandas.values).all())

In [ ]:
# Al terminar, parar PostgreSQL
!docker compose -f ../docker-compose.yml down